# 04a — Build the held-out math evaluation set

Produces the JSONL file that `04_eval_math.ipynb` reads via `HELDOUT_PATH`.

**Run this before notebook 04.** No GPU needed. Needs internet for the
Hugging Face sources (there is also an offline path — see below).

## What it does

1. Pulls candidate items from one or more Bengali math benchmarks
2. Normalises them all to `{id, problem, answer, source, difficulty}`
3. **Removes anything overlapping your training set** — by ID, by exact text,
   and by translation-robust numeric fingerprint
4. Drops items whose gold answer is a worked solution rather than a final
   answer, because notebook 04 scores by exact match
5. Optionally draws a stratified subsample
6. Writes `heldout_math_eval.jsonl` plus a manifest and a composition table

## What it does *not* do

It does not invent questions or answers. Every item comes from a real published
benchmark or from held-out rows of the Ganit dataset. If a source fails to load,
the notebook says so and continues with the others rather than quietly
substituting anything.

## ⚠️ The overlap problem — why this notebook exists

Your elicitation set was carved out of Ganit's SFT split. If your evaluation
items came from the same place, "after LoRA" accuracy measures memorisation, and
both H1a and H1b become meaningless.

Contamination here is not always obvious. Ganit is largely machine-translated
from English sources, so the *same* underlying problem can appear in training
and evaluation with almost no Bengali string overlap — two different
translations of one NuminaMath item look like different problems to any n-gram
check.

This notebook therefore uses three signals, strongest first:

| Signal | Catches |
|---|---|
| **ID match** | the same Ganit row appearing on both sides |
| **Exact normalised text** | identical wording after digit/punctuation normalisation |
| **Numeric fingerprint + token Jaccard** | re-translations — the *numbers* in a word problem survive translation even when every word changes |

Anything flagged is removed and logged, so you can report the count.

## 1. Configuration

In [1]:
# ---- where your TRAINING data lives (required, for the overlap check) ------
TRAIN_DATA_PATH =  r"C:\Users\PC\Downloads\curated\ganit_limo_n1000_c1-16_seed42.jsonl"   # <<< FILL THIS IN
                                                     # same file as DATA_PATH in 01/02/03

# ---- output ----------------------------------------------------------------
OUTPUT_PATH = "heldout_math_eval.jsonl"   # point HELDOUT_PATH in notebook 04 here
OUTPUT_DIR = "heldout"                    # manifest + per-source files land here

# ---- which sources to draw from --------------------------------------------
# Remove any you do not want. If a source fails to load, the notebook reports it
# and carries on with the rest.
#
#   "ganit_dev"     Ganit's own dev split. Native Bengali, same distribution as
#                   training, difficulty-tagged. The closest match to what you
#                   trained on -- and therefore the fairest test of elicitation.
#   "ganit_holdout" Rows of Ganit SFT that are NOT in your training set.
#                   Guaranteed disjoint by ID. Use if the dev split will not load.
#   "bn_mgsm"       Bengali MGSM. 250 grade-school items, translated from GSM8K.
#   "bennumeval"    Native Bengali numerical reasoning, six task types, MIT.
SOURCES = ["ganit_dev"]

# ---- size ------------------------------------------------------------------
TARGET_N = 500          # None = keep everything that survives filtering.
                        # 300-600 is a sensible range: big enough for a McNemar
                        # test to have some power, small enough that six eval
                        # passes finish in hours rather than days.
STRATIFY_BY = "source"  # "source", "difficulty", or None

# ---- filtering -------------------------------------------------------------
MAX_GOLD_ANSWER_CHARS = 60   # longer than this is probably a worked solution,
                             # not a final answer -- exact match would score ~0
MIN_PROBLEM_CHARS = 15
DEDUP_JACCARD = 0.80         # within the held-out set itself
CONTAM_JACCARD = 0.70        # against the training set

SEED = 42

In [2]:
import json, re, hashlib, random, csv, collections
from pathlib import Path
from datetime import datetime, timezone

random.seed(SEED)

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Output file      : {Path(OUTPUT_PATH).resolve()}")
print(f"Manifest dir     : {out_dir.resolve()}")
print(f"Sources requested: {SOURCES}")
print(f"Seed             : {SEED}")

Output file      : C:\Users\PC\791\heldout_math_eval.jsonl
Manifest dir     : C:\Users\PC\791\heldout
Sources requested: ['ganit_dev']
Seed             : 42


## 2. Text normalisation helpers

Ganit stores numbers in both Bengali and Western digits depending on the source
corpus, so everything is mapped to Western digits before any comparison.

In [3]:
BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
NUMBER_RE = re.compile(r"\d+(?:\.\d+)?")


def normalise(text):
    """Lowercase, Bengali digits -> Western, strip punctuation, collapse space."""
    if not text:
        return ""
    t = str(text).translate(BN_DIGITS).lower()
    t = re.sub(r"[^\w\s.]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def numeric_fingerprint(text):
    """MD5 of the sorted set of numbers in a problem.

    This is the signal that catches re-translations. Two Bengali renderings of
    the same English word problem share almost no tokens but contain the same
    numbers. Deliberately loose -- it over-flags, which is the safe direction
    for a contamination check.
    """
    nums = NUMBER_RE.findall(normalise(text))
    if not nums:
        return None
    canon = sorted(str(float(n)).rstrip("0").rstrip(".") for n in nums)
    return hashlib.md5("|".join(canon).encode()).hexdigest()


def token_set(text):
    return set(normalise(text).split())


def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def looks_like_final_answer(ans):
    """A final answer is short and usually numeric. A full worked solution is
    neither, and would score 0 under exact match no matter how good the model."""
    s = str(ans).strip()
    if not s or len(s) > MAX_GOLD_ANSWER_CHARS:
        return False
    return len(s.split()) <= 8


# self-test
assert numeric_fingerprint("রহিম ৫টি আম কিনল ৩ টাকায়") == numeric_fingerprint("Rahim bought 5 mangoes for 3 taka")
print("PASS  numeric fingerprint matches across Bengali/English digit forms")
print(f"PASS  gold-answer filter: '42' -> {looks_like_final_answer('42')}, "
      f"long solution -> {looks_like_final_answer('x'*200)}")

PASS  numeric fingerprint matches across Bengali/English digit forms
PASS  gold-answer filter: '42' -> True, long solution -> False


## 3. Load the training set

Required. Without it there is no overlap check, and an unchecked held-out set is
not worth building.

In [ ]:
train_path = Path(TRAIN_DATA_PATH)
if not train_path.exists():
    raise FileNotFoundError(
        f"TRAIN_DATA_PATH does not exist: {train_path}\n\n"
        "This is required, not optional. The whole point of this notebook is to\n"
        "guarantee the eval set is disjoint from what you trained on, and that\n"
        "cannot be checked without the training file. Set it in the config cell."
    )

train_rows = [json.loads(l) for l in open(train_path, encoding="utf-8") if l.strip()]
train_ids = {str(r.get("id")) for r in train_rows if r.get("id") is not None}
train_norm_text = {normalise(r.get("problem", "")) for r in train_rows}

train_fp = collections.defaultdict(list)
for r in train_rows:
    fp = numeric_fingerprint(r.get("problem", ""))
    if fp:
        train_fp[fp].append(token_set(r.get("problem", "")))

print(f"Training rows          : {len(train_rows)}")
print(f"Training IDs indexed   : {len(train_ids)}")
print(f"Numeric fingerprints   : {len(train_fp)}")

## 4. Load candidate items

Each loader is wrapped so one failure does not stop the rest. Field names on the
Ganit hub could not be verified in advance, so those loaders detect columns and
report what they found instead of assuming.

In [4]:
candidates = []
load_report = {}


def _add(items, source):
    kept = 0
    for it in items:
        problem = str(it.get("problem") or "").strip()
        answer = str(it.get("answer") or "").strip()
        if len(problem) < MIN_PROBLEM_CHARS or not answer:
            continue
        candidates.append({
            "id": f"{source}-{it.get('id', kept)}",
            "problem": problem,
            "answer": answer,
            "source": source,
            "difficulty": it.get("difficulty"),
            "origin_id": it.get("id"),
        })
        kept += 1
    load_report[source] = kept
    print(f"  {source:<16} {kept} items")


def pick(colnames, options):
    for o in options:
        if o in colnames:
            return o
    return None

In [5]:
# ---------- Ganit dev split -------------------------------------------------
if "ganit_dev" in SOURCES:
    try:
        from datasets import load_dataset, get_dataset_config_names
        configs = get_dataset_config_names("dipta007/Ganit")
        print(f"Ganit configs on the hub: {configs}")

        dev_cfg = pick(configs, ["dev", "DEV", "Ganit-Dev", "GanitDEV", "GanitDev"])
        if dev_cfg is None:
            print("  Could not identify a dev config. Skipping ganit_dev.")
            print("  Add the correct name to the pick() list above if you see it "
                  "in the list printed.")
            load_report["ganit_dev"] = 0
        else:
            ds = load_dataset("dipta007/Ganit", dev_cfg)
            split = list(ds.keys())[0]
            rows = ds[split]
            cols = rows.column_names
            print(f"  using config '{dev_cfg}', split '{split}', columns: {cols}")

            qcol = pick(cols, ["problem", "question", "Question"])
            acol = pick(cols, ["bengali_solution", "answer", "Answer", "solution"])
            dcol = pick(cols, ["difficulty", "level"])
            if qcol is None or acol is None:
                print(f"  Could not find question/answer columns in {cols}. Skipping.")
                load_report["ganit_dev"] = 0
            else:
                print(f"  question='{qcol}'  answer='{acol}'  difficulty='{dcol}'")
                _add([{"id": r.get("id", i), "problem": r[qcol], "answer": r[acol],
                       "difficulty": r.get(dcol) if dcol else None}
                      for i, r in enumerate(rows)], "ganit_dev")
    except Exception as e:
        print(f"  ganit_dev FAILED to load: {str(e)[:200]}")
        load_report["ganit_dev"] = 0

Ganit configs on the hub: ['RLVR', 'SFT', 'dev']


Generating dev split:   0%|          | 0/776 [00:00<?, ? examples/s]

  using config 'dev', split 'dev', columns: ['problem', 'source_name', 'id', 'bengali_solution', 'english_solution', 'correct_counts', 'difficulty', 'messages', 'deepseek_outputs', 'gpt_outputs', 'gemini_outputs', 'grok_outputs', 'valid']
  question='problem'  answer='bengali_solution'  difficulty='difficulty'
  ganit_dev        776 items


In [6]:
# ---------- Ganit SFT rows not used in training -----------------------------
# Guaranteed disjoint by ID. Useful when the dev split will not load.
if "ganit_holdout" in SOURCES:
    try:
        from datasets import load_dataset
        ds = load_dataset("dipta007/Ganit", "SFT")
        rows = ds[list(ds.keys())[0]]
        cols = rows.column_names
        print(f"Ganit SFT columns: {cols}")

        qcol = pick(cols, ["problem", "question"])
        acol = pick(cols, ["bengali_solution", "answer", "solution"])
        held = []
        for i, r in enumerate(rows):
            rid = r.get("id", i)
            if str(rid) in train_ids:
                continue                      # was used for training
            held.append({"id": rid, "problem": r[qcol], "answer": r[acol],
                         "difficulty": r.get("difficulty"),
                         "correct_counts": r.get("correct_counts")})
        print(f"  {len(held)} SFT rows are not in your training set")
        _add(held, "ganit_holdout")
    except Exception as e:
        print(f"  ganit_holdout FAILED to load: {str(e)[:200]}")
        load_report["ganit_holdout"] = 0

In [7]:
# ---------- Bn-MGSM ---------------------------------------------------------
# juletxara/mgsm, config 'bn', split 'test'. In the test split, 'answer' and
# 'equation_solution' are null; only 'question' and 'answer_number' carry data.
if "bn_mgsm" in SOURCES:
    try:
        from datasets import load_dataset
        ds = load_dataset("juletxara/mgsm", "bn", split="test")
        print(f"Bn-MGSM columns: {ds.column_names}")
        _add([{"id": i, "problem": r["question"], "answer": r["answer_number"],
               "difficulty": "grade_school"}
              for i, r in enumerate(ds)], "bn_mgsm")
    except Exception as e:
        print(f"  bn_mgsm FAILED to load: {str(e)[:200]}")
        load_report["bn_mgsm"] = 0

In [8]:
# ---------- BenNumEval ------------------------------------------------------
# ka05ar/BenNumEval, MIT, ungated. Six task configs; column names vary a little
# between them, so each is detected rather than assumed.
if "bennumeval" in SOURCES:
    try:
        from datasets import load_dataset
        collected, per_task = [], {}
        for cfg in ["CA", "DS", "CQ", "FiB", "QNLI", "AWP"]:
            try:
                d = load_dataset("ka05ar/BenNumEval", cfg, split="test")
                qcol = pick(d.column_names, ["Question", "question", "Problem", "problem"])
                acol = pick(d.column_names, ["Answer", "answer", "Label", "label"])
                if qcol is None or acol is None:
                    print(f"  BenNumEval[{cfg}]: columns {d.column_names} "
                          "not recognised, skipped")
                    continue
                for i, r in enumerate(d):
                    collected.append({"id": f"{cfg}-{i}", "problem": r[qcol],
                                      "answer": r[acol], "difficulty": cfg})
                per_task[cfg] = len(d)
            except Exception as e:
                print(f"  BenNumEval[{cfg}] failed: {str(e)[:120]}")
        print(f"  per task: {per_task}")
        _add(collected, "bennumeval")
    except Exception as e:
        print(f"  bennumeval FAILED to load: {str(e)[:200]}")
        load_report["bennumeval"] = 0

In [9]:
print(f"\n{'='*58}")
print(f"Candidate items loaded: {len(candidates)}")
for s, n in load_report.items():
    print(f"  {s:<18} {n}")
print(f"{'='*58}")

if not candidates:
    raise RuntimeError(
        "No candidate items loaded from any source.\n"
        "Check your internet connection and the messages above. Nothing was "
        "substituted -- this notebook will not fabricate an evaluation set."
    )


Candidate items loaded: 776
  ganit_dev          776


## 5. Filter out unusable items

Notebook 04 scores by exact match on a final answer. Two kinds of item break
that and would depress every model's score equally, telling you nothing:

- gold answers that are full worked solutions rather than a final value
- duplicated problems inside the held-out set itself

In [10]:
before = len(candidates)

# gold must look like a final answer
bad_gold = [c for c in candidates if not looks_like_final_answer(c["answer"])]
candidates = [c for c in candidates if looks_like_final_answer(c["answer"])]
print(f"Dropped {len(bad_gold)} items whose gold answer looks like a worked "
      f"solution (> {MAX_GOLD_ANSWER_CHARS} chars or > 8 words)")
if bad_gold:
    print("  example:", repr(bad_gold[0]["answer"][:120]))

# internal deduplication
seen_fp, deduped, dropped_dupes = {}, [], 0
for c in candidates:
    fp = numeric_fingerprint(c["problem"])
    toks = token_set(c["problem"])
    is_dupe = False
    if fp and fp in seen_fp:
        for other in seen_fp[fp]:
            if jaccard(toks, other) >= DEDUP_JACCARD:
                is_dupe = True
                break
    if is_dupe:
        dropped_dupes += 1
    else:
        deduped.append(c)
        if fp:
            seen_fp.setdefault(fp, []).append(toks)
candidates = deduped

print(f"Dropped {dropped_dupes} near-duplicates within the held-out pool")
print(f"\n{before} -> {len(candidates)} candidates remain")

Dropped 0 items whose gold answer looks like a worked solution (> 60 chars or > 8 words)
Dropped 0 near-duplicates within the held-out pool

776 -> 776 candidates remain


## 6. Decontaminate against the training set

The important cell. Every removal is logged with the reason, so you can report
the number honestly in the paper.

In [12]:
import json
import csv
import collections
from pathlib import Path
from datasets import load_dataset

# ============================================================
# 1. ACTUAL TRAINING DATA
# ============================================================


with open(TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    train_rows = [json.loads(line) for line in f]

print(f"Training examples: {len(train_rows)}")


# ============================================================
# 2. BUILD CONTAMINATION REFERENCE POOL
# ============================================================

contaminants = []

# ---------- Bn-MGSM ----------
try:
    mgsm = load_dataset(
        "juletxara/mgsm",
        "bn",
        split="test"
    )

    contaminants += [
        ("Bn-MGSM", row["question"])
        for row in mgsm
    ]

    print(f"Bn-MGSM: {len(mgsm)} problems")

except Exception as e:
    print(f"Bn-MGSM FAILED: {e}")


# ---------- Bn-MSVAMP ----------
try:
    msvamp = load_dataset(
        "Mathoctopus/MSVAMP",
        "bn",
        split="test"
    )

    contaminants += [
        ("Bn-MSVAMP", row["m_query"])
        for row in msvamp
    ]

    print(f"Bn-MSVAMP: {len(msvamp)} problems")

except Exception as e:
    print(f"Bn-MSVAMP FAILED: {e}")


# ---------- BenNumEval ----------
bennumeval_configs = ["CA", "DS", "CQ", "FiB", "AWP"]

bennumeval_total = 0

for cfg in bennumeval_configs:
    try:
        ds = load_dataset(
            "ka05ar/BenNumEval",
            cfg,
            split="test"
        )

        text_col = next(
            (
                c for c in [
                    "Question",
                    "question",
                    "Problem",
                    "problem",
                    "Text",
                    "text"
                ]
                if c in ds.column_names
            ),
            None
        )

        if text_col is None:
            print(
                f"BenNumEval[{cfg}]: "
                f"no question column, skipping"
            )
            continue

        contaminants += [
            (f"BenNumEval-{cfg}", row[text_col])
            for row in ds
        ]

        bennumeval_total += len(ds)

    except Exception as e:
        print(
            f"BenNumEval[{cfg}] FAILED: {e}"
        )

print(
    f"BenNumEval: {bennumeval_total} problems"
)

print(
    f"\nTotal contaminant questions: "
    f"{len(contaminants)}"
)


# ============================================================
# 3. BUILD TRAINING INDEXES
# ============================================================

train_ids = {
    str(row["id"])
    for row in train_rows
    if row.get("id") is not None
}

train_norm_text = {
    normalise(row["problem"])
    for row in train_rows
    if row.get("problem")
}

train_fp = collections.defaultdict(list)

for row in train_rows:
    problem = row.get("problem")

    if not problem:
        continue

    fp = numeric_fingerprint(problem)

    if fp:
        train_fp[fp].append(
            token_set(problem)
        )

print("\nTraining indexes:")
print(f"  IDs:         {len(train_ids)}")
print(f"  Text:        {len(train_norm_text)}")
print(f"  Fingerprints:{len(train_fp)}")

Training examples: 873


Bn-MGSM: 250 problems


Bn-MSVAMP: 1000 problems


BenNumEval: 2830 problems

Total contaminant questions: 4080

Training indexes:
  IDs:         873
  Text:        873
  Fingerprints:577


In [13]:
# ============================================================
# 4. DECONTAMINATE HELD-OUT CANDIDATES
# ============================================================

CONTAM_JACCARD = 0.70

kept = []
removed = []

for c in candidates:

    problem = c["problem"]
    hit = None

    # --------------------------------------------------------
    # 1. ID match
    # Only meaningful for Ganit-derived candidates
    # --------------------------------------------------------

    if (
        c.get("origin_id") is not None
        and str(c["origin_id"]) in train_ids
    ):
        hit = (
            "id_match",
            1.0
        )

    # --------------------------------------------------------
    # 2. Exact normalized text
    # --------------------------------------------------------

    if (
        hit is None
        and normalise(problem) in train_norm_text
    ):
        hit = (
            "exact_text",
            1.0
        )

    # --------------------------------------------------------
    # 3. Numeric fingerprint + Jaccard
    #
    # Same numbers alone are NOT enough.
    # We also require token similarity >= 0.70.
    # --------------------------------------------------------

    if hit is None:

        fp = numeric_fingerprint(problem)

        if fp and fp in train_fp:

            toks = token_set(problem)

            best = max(
                (
                    jaccard(toks, train_tokens)
                    for train_tokens in train_fp[fp]
                ),
                default=0.0
            )

            if best >= CONTAM_JACCARD:
                hit = (
                    "numeric_fingerprint+jaccard",
                    round(best, 3)
                )

    # --------------------------------------------------------
    # KEEP / REMOVE
    # --------------------------------------------------------

    if hit:

        removed.append({
            **c,
            "removal_method": hit[0],
            "match_score": hit[1]
        })

    else:
        kept.append(c)


print(
    f"Removed as contaminated : {len(removed)}"
)

print(
    f"Clean items remaining   : {len(kept)}"
)


# ============================================================
# 5. CONTAMINATION REPORT
# ============================================================

if removed:

    by_method = collections.Counter(
        r["removal_method"]
        for r in removed
    )

    by_source = collections.Counter(
        r["source"]
        for r in removed
    )

    print("\nBy removal method:")
    for method, count in by_method.items():
        print(f"  {method}: {count}")

    print("\nBy source:")
    for source, count in by_source.items():
        print(f"  {source}: {count}")


    # Save contamination report
    out_path = Path("removed_contaminated.csv")

    fieldnames = [
        "id",
        "source",
        "removal_method",
        "match_score",
        "problem",
        "answer"
    ]

    with open(
        out_path,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames
        )

        writer.writeheader()

        for r in removed:
            writer.writerow({
                k: r.get(k)
                for k in fieldnames
            })

    print(
        f"\nContamination report saved to: "
        f"{out_path}"
    )

else:

    print(
        "\nNo overlap found."
    )

    print(
        "This is reassuring, but it does not mathematically "
        "prove there is no semantic contamination."
    )


# ============================================================
# 6. SAFETY CHECK
# ============================================================

if not kept:
    raise RuntimeError(
        "Everything was removed as contaminated. "
        "Check your training path and matching thresholds."
    )

Removed as contaminated : 1
Clean items remaining   : 775

By removal method:
  numeric_fingerprint+jaccard: 1

By source:
  ganit_dev: 1

Contamination report saved to: removed_contaminated.csv


## 7. Subsample

Stratified, without replacement. If the pool is smaller than `TARGET_N` you get
the whole pool — it is never padded.

In [14]:
rng = random.Random(SEED)

if TARGET_N is None or TARGET_N >= len(kept):
    final = list(kept)
    if TARGET_N is not None and TARGET_N > len(kept):
        print(f"TARGET_N={TARGET_N} exceeds the clean pool ({len(kept)}). "
              "Keeping everything; the set is not padded.")
elif STRATIFY_BY:
    strata = collections.defaultdict(list)
    for c in kept:
        strata[c.get(STRATIFY_BY)].append(c)
    picked, leftover = [], []
    for key, group in sorted(strata.items(), key=lambda kv: str(kv[0])):
        rng.shuffle(group)
        take = min(len(group), max(1, round(TARGET_N * len(group) / len(kept))))
        picked += group[:take]
        leftover += group[take:]
    rng.shuffle(leftover)
    while len(picked) < TARGET_N and leftover:
        picked.append(leftover.pop())
    # proportional rounding can overshoot; shuffle before trimming so the trim
    # does not systematically empty whichever stratum sorts last
    rng.shuffle(picked)
    final = picked[:TARGET_N]
else:
    shuffled = list(kept)
    rng.shuffle(shuffled)
    final = shuffled[:TARGET_N]

print(f"Final held-out set: {len(final)} items\n")
for key in ["source", "difficulty"]:
    counts = collections.Counter(str(c.get(key)) for c in final)
    print(f"  by {key}:")
    for k, v in counts.most_common():
        print(f"    {k:<24} {v:5d}  {v/len(final):6.1%}")
    print()

Final held-out set: 500 items

  by source:
    ganit_dev                  500  100.0%

  by difficulty:
    easy                       145   29.0%
    hard                       129   25.8%
    medium                     127   25.4%
    olympiad                    99   19.8%



## 8. Verify before writing

Cheap checks that catch the errors which are expensive to discover later.

In [15]:
ids = [c["id"] for c in final]
problems = [c["problem"] for c in final]

checks = {
    "IDs are unique": len(set(ids)) == len(ids),
    "no empty problems": all(str(c["problem"]).strip() for c in final),
    "no empty answers": all(str(c["answer"]).strip() for c in final),
    "no exact duplicate problems": len(set(problems)) == len(problems),
    "no training IDs present": not (set(str(c.get("origin_id")) for c in final) & train_ids),
    "no exact text overlap with training": not any(
        normalise(p) in train_norm_text for p in problems),
    "all gold answers look final": all(looks_like_final_answer(c["answer"]) for c in final),
}

for label, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {label}")

if not all(checks.values()):
    raise AssertionError("A verification check failed. Not writing the file.")

gold_lens = sorted(len(str(c["answer"])) for c in final)
print(f"\nGold answer length: median={gold_lens[len(gold_lens)//2]} "
      f"max={gold_lens[-1]} chars")
numeric = sum(1 for c in final
              if re.fullmatch(r"-?[\d.,\s]+", str(c["answer"]).translate(BN_DIGITS)))
print(f"Purely numeric gold answers: {numeric}/{len(final)} "
      f"({numeric/len(final):.1%})")
if numeric / len(final) < 0.7:
    print("\nNOTE  A sizeable share of answers are not plain numbers. Exact match")
    print("      is stricter on those -- expect some scoring noise from formatting")
    print("      rather than reasoning. Worth a spot-check of the saved file.")

  [PASS] IDs are unique
  [PASS] no empty problems
  [PASS] no empty answers
  [PASS] no exact duplicate problems
  [PASS] no training IDs present
  [PASS] no exact text overlap with training
  [PASS] all gold answers look final

Gold answer length: median=2 max=8 chars
Purely numeric gold answers: 500/500 (100.0%)


## 9. Write the files

In [17]:
# Main file -- the exact schema 04_eval_math.ipynb expects
train_path = Path(TRAIN_DATA_PATH)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for c in final:
        f.write(json.dumps({
            "id": c["id"],
            "problem": c["problem"],
            "answer": str(c["answer"]),
            "source": c["source"],
            "difficulty": c.get("difficulty"),
        }, ensure_ascii=False) + "\n")

# Per-source files, so you can also report accuracy broken down by benchmark
by_source = collections.defaultdict(list)
for c in final:
    by_source[c["source"]].append(c)
for src, items in by_source.items():
    p = out_dir / f"heldout_{src}.jsonl"
    with open(p, "w", encoding="utf-8") as f:
        for c in items:
            f.write(json.dumps({
                "id": c["id"], "problem": c["problem"],
                "answer": str(c["answer"]), "source": c["source"],
                "difficulty": c.get("difficulty"),
            }, ensure_ascii=False) + "\n")
    print(f"  wrote {p}  ({len(items)} items)")

manifest = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "output_path": str(Path(OUTPUT_PATH).resolve()),
    "n_items": len(final),
    "seed": SEED,
    "sources_requested": SOURCES,
    "items_loaded_per_source": load_report,
    "target_n": TARGET_N,
    "stratified_by": STRATIFY_BY,
    "sampled_with_replacement": False,
    "padded_to_target": False,
    "filters": {
        "max_gold_answer_chars": MAX_GOLD_ANSWER_CHARS,
        "min_problem_chars": MIN_PROBLEM_CHARS,
        "internal_dedup_jaccard": DEDUP_JACCARD,
        "contamination_jaccard": CONTAM_JACCARD,
    },
    "decontamination": {
        "training_file": str(train_path),
        "training_rows": len(train_rows),
        "removed_total": len(removed),
        "removed_by_method": dict(collections.Counter(
            r["removal_method"] for r in removed)),
        "methods": ["id_match", "exact_text", "numeric_fingerprint+jaccard"],
        "limitation": ("No upstream English source IDs are available for Ganit, "
                       "so fingerprint and Jaccard matching are proxies. "
                       "Paraphrases may survive."),
    },
    "composition": {
        "by_source": dict(collections.Counter(c["source"] for c in final)),
        "by_difficulty": dict(collections.Counter(
            str(c.get("difficulty")) for c in final)),
    },
    "verification": checks,
}
with open(out_dir / "heldout_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

with open(out_dir / "heldout_composition.csv", "w", newline="",
          encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["source", "n", "share"])
    for k, v in collections.Counter(c["source"] for c in final).most_common():
        w.writerow([k, v, round(v / len(final), 4)])

print(f"\n  wrote {OUTPUT_PATH}  ({len(final)} items)")
print(f"  wrote {out_dir / 'heldout_manifest.json'}")
print(f"  wrote {out_dir / 'heldout_composition.csv'}")

  wrote heldout\heldout_ganit_dev.jsonl  (500 items)

  wrote heldout_math_eval.jsonl  (500 items)
  wrote heldout\heldout_manifest.json
  wrote heldout\heldout_composition.csv


## 10. Preview

In [18]:
print("First three items as written:\n")
with open(OUTPUT_PATH, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        r = json.loads(line)
        print(f"[{r['id']}]  source={r['source']}  difficulty={r['difficulty']}")
        print(f"  problem: {r['problem'][:220]}")
        print(f"  answer : {r['answer']!r}")
        print()

print("=" * 58)
print("NEXT STEP")
print("=" * 58)
print("In 04_eval_math.ipynb, set:")
print(f'    HELDOUT_PATH    = "{OUTPUT_PATH}"')
print(f'    TRAIN_DATA_PATH = "{TRAIN_DATA_PATH}"')
print("\nThe overlap check in notebook 04 will re-verify this independently.")
print("It should report zero overlap. If it does not, something is wrong -- ")
print("stop and investigate rather than proceeding.")

First three items as written:

[ganit_dev-104004]  source=ganit_dev  difficulty=medium
  problem: \( ৫^{২০১৯} - ৩^{২০১৯} \) সমান পূর্ণসংখ্যার এককের অঙ্ক কী?
  answer : '৮'

[ganit_dev-69107]  source=ganit_dev  difficulty=hard
  problem: তাতিয়ানা তিমোফিভানার বয়স ৭২ বছর, ৭২ মাস, ৭২ সপ্তাহ, ৭২ দিন এবং ৭২ ঘন্টা। তাতিয়ানা তিমোফিভানা কত বছর পূর্ণ বয়সী?
  answer : '৭৯'

[ganit_dev-104864]  source=ganit_dev  difficulty=medium
  problem: যখন $১২!$ কে বেস ৪-এ লেখা হয়, তখন $১২!$ কতগুলো শূন্য দিয়ে শেষ হয়?
  answer : '৫'

NEXT STEP
In 04_eval_math.ipynb, set:
    HELDOUT_PATH    = "heldout_math_eval.jsonl"
    TRAIN_DATA_PATH = "C:\Users\PC\Downloads\curated\ganit_limo_n1000_c1-16_seed42.jsonl"

The overlap check in notebook 04 will re-verify this independently.
It should report zero overlap. If it does not, something is wrong -- 
stop and investigate rather than proceeding.


---

## Reading the composition honestly

Two points that belong in your paper's evaluation section:

**Mixed native and translated sources.** Bn-MGSM is translated from GSM8K;
BenNumEval and Ganit-dev are native Bengali. Models can behave quite differently
on the two — translated benchmarks sometimes retain English-friendly phrasing
patterns. The `source` field is preserved in the output and per-source files are
written, so report accuracy broken down by source as well as overall rather than
letting one average hide the difference.

**Difficulty is not comparable across sources.** Ganit's tiers, BenNumEval's task
types and MGSM's grade-school level are different scales wearing the same column
name. Do not aggregate across them.

**On the contamination result.** Zero removals is reassuring but not proof.
Exact and fingerprint matching cannot catch a paraphrase or a different
translation with different numbers. Report the method and the count, and state
the limitation — that is what makes the claim credible rather than the number
itself.

## If you would rather use a single source

Set `SOURCES = ["ganit_dev"]` and `STRATIFY_BY = "difficulty"`. That gives the
cleanest experiment: same distribution as training, native Bengali, real
difficulty tiers, and the fairest possible test of whether LoRA elicitation
transferred. The trade-off is that it says nothing about generalisation beyond
Ganit's own distribution.